In [1]:
from __future__ import annotations
import csv, os, glob, warnings, json
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.utils import resample as sk_resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ── PARAMETERS ────────────────────────────────────────────────────────────────

DATA_ROOT  = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
OUTPUT_DIR = "./resnet50_output"
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
L2_VALUE   = 1e-4

BG        = "#F7F9FC"
TITLE_COL = "#1C2B3A"
SUB_COL   = "#4A5568"

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"TF {tf.__version__} | GPUs: {len(gpus)}")


# ── PREPROCESSING ─────────────────────────────────────────────────────────────

def smart_crop_to_body(image_array: np.ndarray,
                       bg_threshold: float = 15.0,
                       padding_frac: float = 0.03) -> np.ndarray:
    H, W  = image_array.shape[:2]
    gray  = image_array.mean(axis=-1)
    body  = gray > bg_threshold
    rows, cols = np.any(body, axis=1), np.any(body, axis=0)
    if not rows.any() or not cols.any():
        return image_array
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pr = max(2, int((rmax - rmin) * padding_frac))
    pc = max(2, int((cmax - cmin) * padding_frac))
    rmin = max(0, rmin - pr); rmax = min(H - 1, rmax + pr)
    cmin = max(0, cmin - pc); cmax = min(W - 1, cmax + pc)
    crop = image_array[rmin:rmax + 1, cmin:cmax + 1]
    if crop.shape[0] < 10 or crop.shape[1] < 10:
        return image_array
    return cv2.resize(crop.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)


def remove_border_artifacts(image: np.ndarray, crop_fraction: float = 0.02) -> np.ndarray:
    H, W = image.shape[:2]
    ch = max(1, int(H * crop_fraction))
    cw = max(1, int(W * crop_fraction))
    return cv2.resize(image[ch:H - ch, cw:W - cw], (W, H),
                      interpolation=cv2.INTER_LINEAR).astype(np.float32)


def full_preprocessing(image_array: np.ndarray) -> np.ndarray:
    return remove_border_artifacts(
        smart_crop_to_body(image_array, bg_threshold=15.0, padding_frac=0.03),
        crop_fraction=0.02)


# ── DATA LOADING ──────────────────────────────────────────────────────────────

def _collect_files(directory: str, classes: List[str]) -> pd.DataFrame:
    records = []
    for cls in classes:
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fp in glob.glob(os.path.join(cls_dir, "**", "*"), recursive=True):
            if fp.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append({"filename": os.path.abspath(fp), "class": cls})
    return pd.DataFrame(records)


def get_loaders(data_root: str):
    classes   = ["NORMAL", "PNEUMONIA"]
    train_dir = os.path.join(data_root, "train")
    val_dir   = os.path.join(data_root, "val")
    test_dir  = os.path.join(data_root, "test")

    df_all = pd.concat([_collect_files(train_dir, classes),
                        _collect_files(val_dir,   classes)], ignore_index=True)
    df_tr, df_vl = train_test_split(df_all, test_size=0.10,
                                     random_state=42, stratify=df_all["class"])
    df_tr = df_tr.reset_index(drop=True)
    df_vl = df_vl.reset_index(drop=True)
    df_te = _collect_files(test_dir, classes)

    aug_idg  = ImageDataGenerator(
        rescale=1.0 / 255, preprocessing_function=full_preprocessing,
        rotation_range=10, horizontal_flip=True,
        zoom_range=0.15, width_shift_range=0.05, height_shift_range=0.05,
        brightness_range=[0.85, 1.15])
    val_idg  = ImageDataGenerator(rescale=1.0 / 255,
                                   preprocessing_function=full_preprocessing)
    test_idg = ImageDataGenerator(rescale=1.0 / 255,
                                   preprocessing_function=full_preprocessing)

    kw = dict(x_col="filename", y_col="class", directory=None,
              target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode="binary", classes=classes, interpolation="bilinear")

    train_gen = aug_idg.flow_from_dataframe(df_tr, shuffle=True,  **kw)
    val_gen   = val_idg.flow_from_dataframe(df_vl, shuffle=False, **kw)
    test_gen  = test_idg.flow_from_dataframe(df_te, shuffle=False, **kw)

    steps = int(np.ceil(len(df_tr) / BATCH_SIZE))
    print(f"Train={len(df_tr)}  Val={len(df_vl)}  Test={len(df_te)}")
    print(f"Class map: {train_gen.class_indices}")
    return train_gen, val_gen, test_gen, classes, steps


def compute_class_weights_from_gen(train_gen) -> Dict[int, float]:
    w  = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_gen.classes)
    cw = {0: float(w[0]), 1: float(w[1])}
    print(f"Class weights: NORMAL={cw[0]:.3f}  PNEUMONIA={cw[1]:.3f}")
    return cw


# ── MODEL ─────────────────────────────────────────────────────────────────────

def _rebuild_base_with_l2(base_model: keras.Model, l2_val: float) -> keras.Model:
    reg_cfg = keras.regularizers.serialize(keras.regularizers.l2(l2_val))
    raw_cfg = json.loads(base_model.to_json())

    def _inject(node) -> None:
        if isinstance(node, dict):
            cn = node.get("class_name", "")
            if (cn == "Conv2D" or cn.endswith(".Conv2D")) and "config" in node:
                node["config"]["kernel_regularizer"] = reg_cfg
            for v in node.values():
                _inject(v)
        elif isinstance(node, list):
            for item in node:
                _inject(item)

    _inject(raw_cfg)
    rebuilt = keras.models.model_from_json(json.dumps(raw_cfg))
    rebuilt.set_weights(base_model.get_weights())
    return rebuilt


def build_resnet50() -> keras.Model:
    L2_REG    = keras.regularizers.l2(L2_VALUE)
    base_orig = ResNet50(weights="imagenet", include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base = _rebuild_base_with_l2(base_orig, l2_val=L2_VALUE)
    base.trainable = True
    x   = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x   = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(1, activation=None, name="predictions",
                       kernel_regularizer=L2_REG)(x)
    return keras.Model(inputs=base.input, outputs=out, name="ResNet50_Pneumonia")


def train_model(model, train_gen, val_gen, steps_per_epoch,
                class_weight_dict) -> Tuple[keras.Model, Dict]:
    ckpt_path = os.path.join(OUTPUT_DIR, "best_model.keras")
    val_steps = int(np.ceil(val_gen.n / BATCH_SIZE))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=["accuracy", keras.metrics.AUC(name="auc")])
    callbacks = [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                         save_best_only=True, mode="min", verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                           patience=5, min_lr=1e-7, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)]
    h = model.fit(train_gen, steps_per_epoch=steps_per_epoch,
                  validation_data=val_gen, validation_steps=val_steps,
                  epochs=EPOCHS, class_weight=class_weight_dict,
                  callbacks=callbacks, verbose=1)
    return keras.models.load_model(ckpt_path), h.history


# ── EVALUATION ────────────────────────────────────────────────────────────────

def _find_threshold_on_val(model: keras.Model, val_gen) -> float:
    val_gen.reset()
    val_labels, val_probs = [], []
    for _ in range(int(np.ceil(val_gen.n / BATCH_SIZE))):
        imgs, lbls = next(val_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        val_labels.extend(lbls.ravel().astype(int).tolist())
        val_probs.extend(probs.tolist())
    val_labels = np.array(val_labels)
    val_probs  = np.array(val_probs)
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)
    valid   = np.where(tpr >= 0.90)[0]
    opt_idx = valid[np.argmin(fpr[valid])] if len(valid) else np.argmax(tpr)
    thr     = float(thresholds[opt_idx])
    print(f"Validation threshold (sensitivity ≥ 0.90): {thr:.4f}")
    return thr


def evaluate_model(model: keras.Model,
                   val_gen,
                   test_gen) -> Tuple[Dict, float]:
    opt_thr = _find_threshold_on_val(model, val_gen)

    test_gen.reset()
    all_labels, all_probs = [], []
    for _ in range(int(np.ceil(test_gen.n / BATCH_SIZE))):
        imgs, lbls = next(test_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        all_labels.extend(lbls.ravel().astype(int).tolist())
        all_probs.extend(probs.tolist())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    auc        = roc_auc_score(all_labels, all_probs)
    fpr_c, tpr_c, _ = roc_curve(all_labels, all_probs)

    preds = (all_probs >= opt_thr).astype(int)
    TP = int(((preds == 1) & (all_labels == 1)).sum())
    TN = int(((preds == 0) & (all_labels == 0)).sum())
    FP = int(((preds == 1) & (all_labels == 0)).sum())
    FN = int(((preds == 0) & (all_labels == 1)).sum())
    n    = TP + TN + FP + FN + 1e-12
    acc  = (TP + TN) / n
    sens = TP / (TP + FN + 1e-12)
    spec = TN / (TN + FP + 1e-12)
    ppv  = TP / (TP + FP + 1e-12)
    npv  = TN / (TN + FN + 1e-12)
    f1   = 2 * sens * ppv / (sens + ppv + 1e-12)

    boot_aucs = []
    for i in range(1000):
        lb, pb = sk_resample(all_labels, all_probs, random_state=i)
        if len(np.unique(lb)) < 2:
            continue
        boot_aucs.append(roc_auc_score(lb, pb))
    auc_lo = float(np.percentile(boot_aucs, 2.5))
    auc_hi = float(np.percentile(boot_aucs, 97.5))

    print(f"\n{'='*60}\nTEST RESULTS\n{'='*60}")
    for k, v in [
        ("Threshold (val)", f"{opt_thr:.4f}"),
        ("Accuracy",        f"{acc:.4f}"),
        ("Sensitivity",     f"{sens:.4f}"),
        ("Specificity",     f"{spec:.4f}"),
        ("PPV",             f"{ppv:.4f}"),
        ("NPV",             f"{npv:.4f}"),
        ("F1",              f"{f1:.4f}"),
        ("AUC-ROC",         f"{auc:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")]:
        print(f"  {k:<18}: {v}")
    print("=" * 60)

    metrics = dict(threshold=opt_thr, accuracy=acc, sensitivity=sens,
                   specificity=spec, ppv=ppv, npv=npv, f1=f1,
                   auc=auc, auc_ci_lo=auc_lo, auc_ci_hi=auc_hi,
                   TP=TP, TN=TN, FP=FP, FN=FN)
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as fh:
        json.dump({k: round(float(v), 4) for k, v in metrics.items()}, fh, indent=2)

    # ROC curve
    plt.figure(figsize=(5, 5))
    plt.plot(fpr_c, tpr_c, lw=2, color="royalblue",
             label=f"AUC={auc:.3f} [{auc_lo:.3f}–{auc_hi:.3f}]")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve  |  ResNet-50", fontweight="bold")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.close()

    return metrics, opt_thr


# ── PLOTS ─────────────────────────────────────────────────────────────────────

def plot_training_accuracy(history: Dict) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.get("accuracy", []),     label="Train",
            color="#2196F3", lw=2, marker="o", markersize=3)
    ax.plot(history.get("val_accuracy", []), label="Validation",
            color="#FF5722", lw=2, marker="s", markersize=3)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Accuracy  |  ResNet-50", fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_accuracy.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_loss(history: Dict) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.get("loss", []),     label="Train",
            color="#2196F3", lw=2, marker="o", markersize=3)
    ax.plot(history.get("val_loss", []), label="Validation",
            color="#FF5722", lw=2, marker="s", markersize=3)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("Loss  |  ResNet-50", fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_auc(history: Dict) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.get("auc", []),     label="Train",
            color="#2196F3", lw=2, marker="o", markersize=3)
    ax.plot(history.get("val_auc", []), label="Validation",
            color="#FF5722", lw=2, marker="s", markersize=3)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("AUC", fontsize=12)
    ax.set_title("AUC  |  ResNet-50", fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_auc.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(metrics: Dict) -> None:
    TP, TN = metrics["TP"], metrics["TN"]
    FP, FN = metrics["FP"], metrics["FN"]
    cm     = np.array([[TN, FP], [FN, TP]])
    total  = int(cm.sum())

    cmap_correct = plt.get_cmap("Blues")
    cmap_wrong   = plt.get_cmap("Reds")
    diag_max    = max(int(TN), int(TP), 1)
    offdiag_max = max(int(FP), int(FN), 1)

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor("#F8FAFC")
    ax.set_facecolor("#F8FAFC")

    for i in range(2):
        for j in range(2):
            val  = int(cm[i, j])
            pct  = val / total * 100
            diag = (i == j)
            raw       = val / diag_max if diag else val / offdiag_max
            intensity = float(np.clip(raw * 0.74 + 0.18, 0.18, 0.92))
            colour    = cmap_correct(intensity) if diag else cmap_wrong(intensity)
            txt_col   = "white" if intensity > 0.40 else "#1C2B3A"

            rect = plt.Rectangle([j, 1 - i], 1, 1, color=colour, ec="white", lw=4)
            ax.add_patch(rect)
            ax.text(j + 0.5, 1 - i + 0.56, str(val),
                    ha="center", va="center",
                    fontsize=22, fontweight="bold", color=txt_col, clip_on=False)
            ax.text(j + 0.5, 1 - i + 0.30, f"{pct:.1f}%",
                    ha="center", va="center",
                    fontsize=12, color=txt_col, clip_on=False)
            sem = ("True Neg."  if (i == 0 and j == 0) else
                   "False Pos." if (i == 0 and j == 1) else
                   "False Neg." if (i == 1 and j == 0) else "True Pos.")
            ax.text(j + 0.5, 1 - i + 0.11, sem,
                    ha="center", va="center",
                    fontsize=9, style="italic", color=txt_col, clip_on=False)

    labels = ["NORMAL", "PNEUMONIA"]
    ax.set_xlim(-0.08, 2.08); ax.set_ylim(-0.08, 2.08)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(labels, fontsize=12, fontweight="bold")
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(labels[::-1], fontsize=12, fontweight="bold")
    ax.xaxis.set_label_position("bottom"); ax.xaxis.tick_bottom()
    ax.set_xlabel("Predicted Label", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel("True Label",      fontsize=13, fontweight="bold", labelpad=12)

    legend_patches = [
        mpatches.Patch(color=cmap_correct(0.65), label="Correct prediction"),
        mpatches.Patch(color=cmap_wrong(0.60),   label="Incorrect prediction"),
    ]
    ax.legend(handles=legend_patches, loc="upper center",
              bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=10, frameon=False)

    acc = (TP + TN) / (total + 1e-9)
    ax.set_title(
        f"Confusion Matrix  |  ResNet-50\n"
        f"Accuracy: {acc:.4f}  ({TP + TN} / {total})",
        fontsize=14, fontweight="bold", color="#1C2B3A", pad=18)

    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
                dpi=200, bbox_inches="tight", facecolor="#F8FAFC")
    plt.close()


def plot_ppv_npv(metrics: Dict) -> None:
    labels = ["PPV", "NPV"]
    values = [metrics["ppv"], metrics["npv"]]
    colors = ["#5C6BC0", "#26A69A"]
    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(labels, values, color=colors, width=0.4,
                  edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=13, fontweight="bold")
    ax.set_ylim([0, 1.15])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("PPV & NPV  |  ResNet-50", fontsize=12, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ppv_npv_bar.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_bar_chart_metrics(metrics: Dict) -> None:
    labels = ["Accuracy", "Sensitivity", "Specificity", "AUC-ROC", "F1 Score"]
    values = [metrics["accuracy"], metrics["sensitivity"],
              metrics["specificity"], metrics["auc"], metrics["f1"]]
    colors = ["#1E88E5", "#43A047", "#FB8C00", "#8E24AA", "#E53935"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, values, color=colors, width=0.55,
                  edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")
    ax.set_ylim([0, 1.12])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Performance Metrics  |  ResNet-50",
                 fontsize=13, fontweight="bold")
    ax.axhline(y=0.9, color="grey", linestyle="--", lw=1, alpha=0.6)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "bar_chart_metrics.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main() -> None:
    print(f"{'='*70}")
    print(f"  ResNet-50  |  Chest X-ray Pneumonia Classification")
    print(f"  EPOCHS={EPOCHS}  LR={LR}  L2={L2_VALUE}")
    print(f"{'='*70}\n")

    train_gen, val_gen, test_gen, _, steps = get_loaders(DATA_ROOT)
    class_weight_dict = compute_class_weights_from_gen(train_gen)

    ckpt = os.path.join(OUTPUT_DIR, "best_model.keras")
    if os.path.exists(ckpt):
        print(f"Loading checkpoint  ←  {ckpt}")
        model   = keras.models.load_model(ckpt)
        hist_path = os.path.join(OUTPUT_DIR, "history.json")
        history = (json.load(open(hist_path)) if os.path.exists(hist_path) else {})
    else:
        model = build_resnet50()
        model.summary(line_length=90)
        model, history = train_model(model, train_gen, val_gen, steps, class_weight_dict)
        with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as fh:
            json.dump(history, fh, indent=2)

    if history:
        plot_training_accuracy(history)
        plot_training_loss(history)
        plot_training_auc(history)

    metrics, opt_thr = evaluate_model(model, val_gen, test_gen)

    plot_confusion_matrix(metrics)
    plot_ppv_npv(metrics)
    plot_bar_chart_metrics(metrics)

    final = {
        **{k: v for k, v in metrics.items() if k not in ("TP", "TN", "FP", "FN")},
        "data_split": "train+val merged → 90/10 stratified; test unchanged",
        "epochs": EPOCHS, "lr": LR, "l2": L2_VALUE}
    with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in final.items()}, f, indent=2)

    print(f"\nAll outputs  →  {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

2026-05-18 09:23:46.033975: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779096226.595663      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779096226.713971      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779096227.823153      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779096227.823197      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779096227.823200      57 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: 2
  ResNet-50  |  Chest X-ray Pneumonia Classification
  EPOCHS=50  LR=0.001  L2=0.0001

Found 4708 validated image filenames belonging to 2 classes.
Found 524 validated image filenames belonging to 2 classes.
Found 624 validated image filenames belonging to 2 classes.
Train=4708  Val=524  Test=624
Class map: {'NORMAL': 0, 'PNEUMONIA': 1}
Class weights: NORMAL=1.939  PNEUMONIA=0.674


I0000 00:00:1779096281.378551      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779096281.386441      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "ResNet50_Pneumonia"

...

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

Epoch 1/50


I0000 00:00:1779096324.365095     132 service.cc:152] XLA service 0x7e3bac0031e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779096324.365130     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779096324.365134     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779096330.739631     132 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779096357.629435     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8310 - auc: 0.8881 - loss: 1.0861
Epoch 1: val_loss improved from inf to 4.33541, saving model to ./resnet50_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 292s 2s/step - accuracy: 0.8313 - auc: 0.8883 - loss: 1.0852 - val_accuracy: 0.7424 - val_auc: 0.5000 - val_loss: 4.3354 - learning_rate: 0.0010
Epoch 2/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 932ms/step - accuracy: 0.9334 - auc: 0.9544 - loss: 0.7574
Epoch 2: val_loss did not improve from 4.33541
148/148 ━━━━━━━━━━━━━━━━━━━━ 148s 997ms/step - accuracy: 0.9334 - auc: 0.9544 - loss: 0.7573 - val_accuracy: 0.7424 - val_auc: 0.5000 - val_loss: 5.1613 - learning_rate: 0.0010
Epoch 3/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 931ms/step - accuracy: 0.9270 - auc: 0.9474 - loss: 0.6971
Epoch 3: val_loss improved from 4.33541 to 2.74650, saving model to ./resnet50_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 150s 1s/step - accuracy: 0.9270 - auc: 0.9475 - loss: 0.6969 - val_accuracy: 0.

In [1]:
"""
ResNet-50 + Spatial Attention Mechanism (SAM) + MRS Pipeline
═══════════════════════════════════════════════════════════════
Base Model      : ResNet-50 (ImageNet pre-trained, L2-regularised, fully fine-tuned)
Interpretability: Spatial Attention Mechanism (SAM)
                  A  = softmax(Conv(F))          [Colin & Surantha, Information 2025, 16, 53]
                  F' = A ⊙ F                    (element-wise multiplication)
MRS             : Ri = Σ(R_pos · M) / (Σ R_pos + ε)
                  [Hedström et al. JMLR 2023]
All model hyper-parameters kept identical to the base ResNet-50 pipeline:
  IMG_SIZE=224  BATCH_SIZE=32  EPOCHS=50  LR=1e-3  L2=1e-4
Visuals produced (mirrored from the ResNet-50 + Grad-CAM pipeline):
  • training_accuracy.png          • training_loss.png
  • roc_curve.png                  • confusion_matrix.png
  • ppv_npv_chart.png              • metrics_bar_chart.png   (includes MRS)
  • mrs_histogram.png              • accuracy_vs_interpretability.png
  • explanation_normal_panel.png   • explanation_pneumonia_panel.png
  • single_normal_{1-5}.png        • single_pneumonia_{1-5}.png
  • sam_attention_maps.png         • final_report.json  /  mrs_per_sample.csv
"""

from __future__ import annotations
import csv, os, glob, warnings, json, zipfile
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from sklearn.model_selection import train_test_split
from sklearn.utils import resample as sk_resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ─────────────────────────────────────────────────────────────────────────────
#  CONFIGURATION  (identical to base ResNet-50 pipeline)
# ─────────────────────────────────────────────────────────────────────────────

DATA_ROOT  = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
OUTPUT_DIR = "./resnet50_sam_output"
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
L2_VALUE   = 1e-4
SMALL      = 1e-9

# SAM hyper-parameters (from paper: Colin & Surantha, 2025)
SAM_KERNEL_SIZE  = 7          # Conv kernel for attention map generation
SAM_FEAT_H       = 7          # ResNet-50 spatial output at 224×224 → 7×7
SAM_FEAT_W       = 7

# Lung-mask parameters (reused from Grad-CAM pipeline)
LUNG_ADAPTIVE_LO = 0.70
LUNG_ADAPTIVE_HI = 0.80
LUNG_DILATE_SIZE = 7
LUNG_CLOSE_SIZE  = 15
LUNG_TOP_PAD     = 0.03
LUNG_BOTTOM_PAD  = 0.11
LUNG_LEFT_PAD    = 0.04
LUNG_RIGHT_PAD   = 0.04

# MRS reference baseline (SAM result from paper Table 3)
MRS_BASELINE = 0.12

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"TF {tf.__version__} | GPUs: {len(gpus)}")


# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING  (identical to base ResNet-50 pipeline)
# ─────────────────────────────────────────────────────────────────────────────

def smart_crop_to_body(image_array: np.ndarray,
                       bg_threshold: float = 15.0,
                       padding_frac: float = 0.03) -> np.ndarray:
    H, W  = image_array.shape[:2]
    gray  = image_array.mean(axis=-1)
    body  = gray > bg_threshold
    rows, cols = np.any(body, axis=1), np.any(body, axis=0)
    if not rows.any() or not cols.any():
        return image_array
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pr = max(2, int((rmax - rmin) * padding_frac))
    pc = max(2, int((cmax - cmin) * padding_frac))
    rmin = max(0, rmin - pr);  rmax = min(H - 1, rmax + pr)
    cmin = max(0, cmin - pc);  cmax = min(W - 1, cmax + pc)
    crop = image_array[rmin:rmax + 1, cmin:cmax + 1]
    if crop.shape[0] < 10 or crop.shape[1] < 10:
        return image_array
    return cv2.resize(crop.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)


def remove_border_artifacts(image: np.ndarray, crop_fraction: float = 0.02) -> np.ndarray:
    H, W = image.shape[:2]
    ch = max(1, int(H * crop_fraction))
    cw = max(1, int(W * crop_fraction))
    return cv2.resize(image[ch:H - ch, cw:W - cw], (W, H),
                      interpolation=cv2.INTER_LINEAR).astype(np.float32)


def full_preprocessing(image_array: np.ndarray) -> np.ndarray:
    return remove_border_artifacts(
        smart_crop_to_body(image_array, bg_threshold=15.0, padding_frac=0.03),
        crop_fraction=0.02)


# ─────────────────────────────────────────────────────────────────────────────
#  LUNG MASK  (reused from Grad-CAM pipeline for MRS)
# ─────────────────────────────────────────────────────────────────────────────

def recover_lung_mask(x_img: np.ndarray,
                      top_pad:    float = LUNG_TOP_PAD,
                      bottom_pad: float = LUNG_BOTTOM_PAD,
                      left_pad:   float = LUNG_LEFT_PAD,
                      right_pad:  float = LUNG_RIGHT_PAD) -> np.ndarray:
    H, W    = x_img.shape[:2]
    gray    = (0.299 * x_img[:, :, 0] + 0.587 * x_img[:, :, 1] + 0.114 * x_img[:, :, 2])
    gray_u8 = (gray * 255.0).clip(0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(gray_u8, (5, 5), 0)

    _, body = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    body    = cv2.morphologyEx(body, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n_l, labels, stats, _ = cv2.connectedComponentsWithStats(body)
    if n_l > 1:
        best = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        body = (labels == best).astype(np.uint8) * 255

    r1 = max(0, int(H * top_pad));    r2 = min(H, int(H * (1.0 - bottom_pad)))
    c1 = max(0, int(W * left_pad));   c2 = min(W, int(W * (1.0 - right_pad)))
    template = np.zeros((H, W), dtype=np.uint8)
    if r2 > r1 and c2 > c1:
        template[r1:r2, c1:c2] = 255

    kd = LUNG_DILATE_SIZE | 1;  kc = LUNG_CLOSE_SIZE | 1
    if kd > 1:
        template = cv2.dilate(template, np.ones((kd, kd), np.uint8))
    if kc > 1:
        template = cv2.morphologyEx(template, cv2.MORPH_CLOSE, np.ones((kc, kc), np.uint8))
    template[:r1, :] = 0

    if body.sum() > 0:
        body_soft = cv2.dilate(body, np.ones((25, 25), np.uint8))
        template  = cv2.bitwise_and(template, body_soft)
    template = np.clip(template, 0, 255).astype(np.uint8)

    coverage = float((template > 0).sum()) / (H * W)
    if coverage < LUNG_ADAPTIVE_LO:
        ek = max(3, int((LUNG_ADAPTIVE_LO - coverage) * (H + W) / 3)) | 1
        template = cv2.dilate(template, np.ones((ek, ek), np.uint8))
        template = np.clip(template, 0, 255).astype(np.uint8)
    elif coverage > LUNG_ADAPTIVE_HI:
        ek = max(3, int((coverage - LUNG_ADAPTIVE_HI) * (H + W) / 3)) | 1
        template = cv2.erode(template, np.ones((ek, ek), np.uint8))

    return (template > 0).astype(np.float32)


def compute_ri(x_img: np.ndarray, raw_attn: np.ndarray) -> Tuple[float, float]:
    """Compute per-sample MRS relevance index using the SAM attention map."""
    mask  = recover_lung_mask(x_img)
    R_pos = np.maximum(raw_attn, 0.0)
    Ri    = float((R_pos * mask).sum() / (R_pos.sum() + SMALL))
    return Ri, float(mask.mean())


# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING  (identical to base ResNet-50 pipeline)
# ─────────────────────────────────────────────────────────────────────────────

def _collect_files(directory: str, classes: List[str]) -> pd.DataFrame:
    records = []
    for cls in classes:
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fp in glob.glob(os.path.join(cls_dir, "**", "*"), recursive=True):
            if fp.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append({"filename": os.path.abspath(fp), "class": cls})
    return pd.DataFrame(records)


def get_loaders(data_root: str):
    classes   = ["NORMAL", "PNEUMONIA"]
    train_dir = os.path.join(data_root, "train")
    val_dir   = os.path.join(data_root, "val")
    test_dir  = os.path.join(data_root, "test")

    df_all = pd.concat([_collect_files(train_dir, classes),
                        _collect_files(val_dir,   classes)], ignore_index=True)
    df_tr, df_vl = train_test_split(df_all, test_size=0.10,
                                    random_state=42, stratify=df_all["class"])
    df_tr = df_tr.reset_index(drop=True)
    df_vl = df_vl.reset_index(drop=True)
    df_te = _collect_files(test_dir, classes)

    aug_idg  = ImageDataGenerator(
        rescale=1.0 / 255, preprocessing_function=full_preprocessing,
        rotation_range=10, horizontal_flip=True,
        zoom_range=0.15, width_shift_range=0.05, height_shift_range=0.05,
        brightness_range=[0.85, 1.15])
    val_idg  = ImageDataGenerator(rescale=1.0 / 255, preprocessing_function=full_preprocessing)
    test_idg = ImageDataGenerator(rescale=1.0 / 255, preprocessing_function=full_preprocessing)

    kw = dict(x_col="filename", y_col="class", directory=None,
              target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode="binary", classes=classes, interpolation="bilinear")

    train_gen = aug_idg.flow_from_dataframe(df_tr, shuffle=True,  **kw)
    val_gen   = val_idg.flow_from_dataframe(df_vl, shuffle=False, **kw)
    test_gen  = test_idg.flow_from_dataframe(df_te, shuffle=False, **kw)

    steps = int(np.ceil(len(df_tr) / BATCH_SIZE))
    print(f"Train={len(df_tr)}  Val={len(df_vl)}  Test={len(df_te)}")
    print(f"Class map: {train_gen.class_indices}")
    return train_gen, val_gen, test_gen, classes, steps


def compute_class_weights_from_gen(train_gen) -> Dict[int, float]:
    w  = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_gen.classes)
    return {0: float(w[0]), 1: float(w[1])}


# ─────────────────────────────────────────────────────────────────────────────
#  SPATIAL ATTENTION MECHANISM (SAM)
#  Implementation: Colin & Surantha, Information 2025, 16, 53 — Section 4.1.1
#    A  = softmax(Conv(F))       where F = ResNet-50 feature maps
#    F' = A ⊙ F                  (element-wise multiplication)
# ─────────────────────────────────────────────────────────────────────────────

class SpatialAttentionModule(layers.Layer):
    """
    Spatial Attention Module (SAM).
    Produces a spatial probability map A over (H×W) feature positions,
    then weights each spatial location of F by its attention score.

    Forward pass:
        x_conv  : (B, H, W, 1)   — single-channel projection
        A_flat  : (B, H*W)        — softmax over all spatial positions
        A       : (B, H, W, 1)   — reshaped attention map (stored for extraction)
        F'      : (B, H, W, C)   — attended feature maps
    """

    def __init__(self, kernel_size: int = SAM_KERNEL_SIZE,
                 l2_val: float = L2_VALUE, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = layers.Conv2D(
            1, kernel_size=kernel_size, padding="same",
            activation=None,
            kernel_regularizer=keras.regularizers.l2(l2_val),
            name="sam_conv_proj")
        self.bn   = layers.BatchNormalization(name="sam_bn")

    def call(self, inputs, training=None):
        B       = tf.shape(inputs)[0]
        H, W, C = inputs.shape[1], inputs.shape[2], inputs.shape[3]

        # Project to single-channel spatial map
        proj    = self.conv(inputs)                          # (B, H, W, 1)
        proj    = self.bn(proj, training=training)

        # Softmax over spatial positions → attention probability map
        flat    = tf.reshape(proj, [B, H * W])               # (B, H*W)
        flat    = tf.nn.softmax(flat, axis=-1)                # sum=1 per sample
        attn    = tf.reshape(flat, [B, H, W, 1])             # (B, H, W, 1)

        # Element-wise weighting: F' = A ⊙ F
        attended = inputs * attn                              # (B, H, W, C)
        return attended, attn                                 # return both

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"kernel_size": self.kernel_size})
        return cfg


# ─────────────────────────────────────────────────────────────────────────────
#  MODEL — ResNet-50 + SAM head
# ─────────────────────────────────────────────────────────────────────────────

def _rebuild_base_with_l2(base_model: keras.Model, l2_val: float) -> keras.Model:
    reg_cfg = keras.regularizers.serialize(keras.regularizers.l2(l2_val))
    raw_cfg = json.loads(base_model.to_json())

    def _inject(node) -> None:
        if isinstance(node, dict):
            cn = node.get("class_name", "")
            if (cn == "Conv2D" or cn.endswith(".Conv2D")) and "config" in node:
                node["config"]["kernel_regularizer"] = reg_cfg
            for v in node.values():
                _inject(v)
        elif isinstance(node, list):
            for item in node:
                _inject(item)

    _inject(raw_cfg)
    rebuilt = keras.models.model_from_json(json.dumps(raw_cfg))
    rebuilt.set_weights(base_model.get_weights())
    return rebuilt


def build_resnet50_sam() -> Tuple[keras.Model, keras.Model]:
    """
    Returns:
        model           — full classification model (trained / saved)
        attn_extractor  — sub-model that returns (attention_map, logit) per image
    """
    L2_REG    = keras.regularizers.l2(L2_VALUE)
    base_orig = ResNet50(weights="imagenet", include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base = _rebuild_base_with_l2(base_orig, l2_val=L2_VALUE)
    base.trainable = True

    # ── SAM ──────────────────────────────────────────────────────────────────
    sam_module  = SpatialAttentionModule(kernel_size=SAM_KERNEL_SIZE,
                                         l2_val=L2_VALUE, name="sam")
    attended, attn_map = sam_module(base.output)             # (B,7,7,2048), (B,7,7,1)

    # ── Classification head (same as base pipeline) ───────────────────────
    x   = layers.GlobalAveragePooling2D(name="gap")(attended)
    x   = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(1, activation=None, name="predictions",
                       kernel_regularizer=L2_REG)(x)

    # ── Full model ────────────────────────────────────────────────────────
    model = keras.Model(inputs=base.input, outputs=out,
                        name="ResNet50_SAM_Pneumonia")

    # ── Attention extractor sub-model ─────────────────────────────────────
    #   Returns attention map (7×7×1) + logit side by side
    attn_extractor = keras.Model(inputs=base.input,
                                 outputs=[attn_map, out],
                                 name="AttentionExtractor")
    return model, attn_extractor


# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def train_model(model: keras.Model, train_gen, val_gen,
                steps_per_epoch, class_weight_dict) -> Tuple[keras.Model, Dict]:
    ckpt_path = os.path.join(OUTPUT_DIR, "best_model.keras")
    val_steps = int(np.ceil(val_gen.n / BATCH_SIZE))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=["accuracy", keras.metrics.AUC(name="auc")])
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True)]
    h = model.fit(train_gen, steps_per_epoch=steps_per_epoch,
                  validation_data=val_gen, validation_steps=val_steps,
                  epochs=EPOCHS, class_weight=class_weight_dict,
                  callbacks=callbacks, verbose=1)
    return keras.models.load_model(ckpt_path, custom_objects={"SpatialAttentionModule": SpatialAttentionModule}), h.history


# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def _find_threshold_on_val(model: keras.Model, val_gen) -> float:
    val_gen.reset()
    val_labels, val_probs = [], []
    for _ in range(int(np.ceil(val_gen.n / BATCH_SIZE))):
        imgs, lbls = next(val_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        val_labels.extend(lbls.ravel().astype(int).tolist())
        val_probs.extend(probs.tolist())

    val_labels = np.array(val_labels)
    val_probs  = np.array(val_probs)
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)

    valid = np.where((tpr >= 0.90) & (thresholds >= 0.88))[0]
    if len(valid) == 0:
        valid = np.where(tpr >= 0.90)[0]
    opt_idx = valid[np.argmin(fpr[valid])] if len(valid) else np.argmax(tpr)
    thr = float(thresholds[opt_idx])
    print(f"Validation threshold (sensitivity ≥ 0.90, thr ≥ 0.50): {thr:.4f}")
    return thr


def evaluate_model(model: keras.Model, val_gen, test_gen) -> Tuple[Dict, float]:
    opt_thr = _find_threshold_on_val(model, val_gen)

    test_gen.reset()
    all_labels, all_probs = [], []
    for _ in range(int(np.ceil(test_gen.n / BATCH_SIZE))):
        imgs, lbls = next(test_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        all_labels.extend(lbls.ravel().astype(int).tolist())
        all_probs.extend(probs.tolist())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    auc        = roc_auc_score(all_labels, all_probs)
    fpr_c, tpr_c, _ = roc_curve(all_labels, all_probs)

    preds = (all_probs >= opt_thr).astype(int)
    TP = int(((preds == 1) & (all_labels == 1)).sum())
    TN = int(((preds == 0) & (all_labels == 0)).sum())
    FP = int(((preds == 1) & (all_labels == 0)).sum())
    FN = int(((preds == 0) & (all_labels == 1)).sum())
    n    = TP + TN + FP + FN + SMALL
    acc  = (TP + TN) / n
    sens = TP  / (TP + FN + SMALL)
    spec = TN  / (TN + FP + SMALL)
    ppv  = TP  / (TP + FP + SMALL)
    npv  = TN  / (TN + FN + SMALL)
    f1   = 2 * sens * ppv / (sens + ppv + SMALL)

    boot_aucs = []
    for i in range(1000):
        lb, pb = sk_resample(all_labels, all_probs, random_state=i)
        if len(np.unique(lb)) < 2:
            continue
        boot_aucs.append(roc_auc_score(lb, pb))
    auc_lo = float(np.percentile(boot_aucs, 2.5))
    auc_hi = float(np.percentile(boot_aucs, 97.5))

    print(f"\n{'='*60}\nTEST RESULTS\n{'='*60}")
    for k, v in [
        ("Threshold (val)", f"{opt_thr:.4f}"),
        ("Accuracy",        f"{acc:.4f}"),
        ("Sensitivity",     f"{sens:.4f}"),
        ("Specificity",     f"{spec:.4f}"),
        ("PPV",             f"{ppv:.4f}"),
        ("NPV",             f"{npv:.4f}"),
        ("F1",              f"{f1:.4f}"),
        ("AUC-ROC",         f"{auc:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")]:
        print(f"  {k:<18}: {v}")
    print("=" * 60)

    metrics = dict(
        threshold=opt_thr, accuracy=acc, sensitivity=sens,
        specificity=spec, ppv=ppv, npv=npv, f1=f1,
        auc=auc, auc_ci_lo=auc_lo, auc_ci_hi=auc_hi,
        TP=TP, TN=TN, FP=FP, FN=FN)
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as fh:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in metrics.items()}, fh, indent=2)

    # ROC curve
    plt.figure(figsize=(5, 5))
    plt.plot(fpr_c, tpr_c, lw=2, color="royalblue",
             label=f"AUC={auc:.3f} [{auc_lo:.3f}-{auc_hi:.3f}]")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve  |  ResNet-50 + SAM", fontweight="bold")
    plt.legend(fontsize=9); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.close()

    return metrics, opt_thr


# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_training_accuracy(history: Dict) -> None:
    epochs = range(1, len(history["accuracy"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history["accuracy"],     color="#2196F3", lw=2, marker="o",
            markersize=3, label="Training Accuracy")
    ax.plot(epochs, history["val_accuracy"], color="#FF5722", lw=2, marker="s",
            markersize=3, label="Validation Accuracy")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Training & Validation Accuracy  |  ResNet-50 + SAM",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_accuracy.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_loss(history: Dict) -> None:
    epochs = range(1, len(history["loss"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history["loss"],     color="#2196F3", lw=2, marker="o",
            markersize=3, label="Training Loss")
    ax.plot(epochs, history["val_loss"], color="#FF5722", lw=2, marker="s",
            markersize=3, label="Validation Loss")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("Training & Validation Loss  |  ResNet-50 + SAM",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"), dpi=150, bbox_inches="tight")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
#  METRIC PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_metrics_bar_chart(metrics: Dict, mrs_overall: float) -> None:
    labels = ["Accuracy", "Sensitivity", "Specificity", "AUC-ROC", "F1 Score", "MRS"]
    values = [metrics["accuracy"], metrics["sensitivity"], metrics["specificity"],
              metrics["auc"], metrics["f1"], mrs_overall]
    colors = ["#1E88E5", "#43A047", "#FB8C00", "#8E24AA", "#E53935", "#00ACC1"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim([0, 1.12])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Model Performance Metrics  |  ResNet-50 + SAM",
                 fontsize=13, fontweight="bold")
    ax.axhline(y=0.9, color="grey", linestyle="--", linewidth=1,
               alpha=0.6, label="0.90 reference")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "metrics_bar_chart.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_ppv_npv(metrics: Dict) -> None:
    labels = ["PPV\n(Precision)", "NPV\n(Neg. Pred. Value)"]
    values = [metrics["ppv"], metrics["npv"]]
    colors = ["#5C6BC0", "#26A69A"]

    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(labels, values, color=colors, width=0.4, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax.set_ylim([0, 1.15])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Predictive Values  |  ResNet-50 + SAM", fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ppv_npv_chart.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(metrics: Dict) -> None:
    TP, TN = metrics["TP"], metrics["TN"]
    FP, FN = metrics["FP"], metrics["FN"]
    cm     = np.array([[TN, FP], [FN, TP]])
    total  = int(cm.sum())

    cmap_correct = plt.get_cmap("Blues")
    cmap_wrong   = plt.get_cmap("Reds")
    diag_max    = max(int(TN), int(TP), 1)
    offdiag_max = max(int(FP), int(FN), 1)

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor("#F8FAFC")
    ax.set_facecolor("#F8FAFC")

    for i in range(2):
        for j in range(2):
            val  = int(cm[i, j])
            pct  = val / total * 100
            diag = (i == j)
            raw       = val / diag_max if diag else val / offdiag_max
            intensity = float(np.clip(raw * 0.74 + 0.18, 0.18, 0.92))
            colour    = cmap_correct(intensity) if diag else cmap_wrong(intensity)
            txt_col   = "white" if intensity > 0.40 else "#1C2B3A"

            rect = plt.Rectangle([j, 1 - i], 1, 1, color=colour, ec="white", lw=4)
            ax.add_patch(rect)
            ax.text(j + 0.5, 1 - i + 0.56, str(val),
                    ha="center", va="center",
                    fontsize=22, fontweight="bold", color=txt_col, clip_on=False)
            ax.text(j + 0.5, 1 - i + 0.30, f"{pct:.1f}%",
                    ha="center", va="center", fontsize=12, color=txt_col, clip_on=False)
            sem = ("True Neg." if (i == 0 and j == 0) else
                   "False Pos." if (i == 0 and j == 1) else
                   "False Neg." if (i == 1 and j == 0) else "True Pos.")
            ax.text(j + 0.5, 1 - i + 0.11, sem,
                    ha="center", va="center",
                    fontsize=9, style="italic", color=txt_col, clip_on=False)

    labels = ["NORMAL", "PNEUMONIA"]
    ax.set_xlim(-0.08, 2.08); ax.set_ylim(-0.08, 2.08)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(labels, fontsize=12, fontweight="bold")
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(labels[::-1], fontsize=12, fontweight="bold")
    ax.xaxis.set_label_position("bottom"); ax.xaxis.tick_bottom()
    ax.set_xlabel("Predicted Label", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel("True Label",      fontsize=13, fontweight="bold", labelpad=12)

    legend_patches = [
        mpatches.Patch(color=cmap_correct(0.65), label="Correct prediction"),
        mpatches.Patch(color=cmap_wrong(0.60),   label="Incorrect prediction"),
    ]
    ax.legend(handles=legend_patches, loc="upper center",
              bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=10, frameon=False)

    acc = (TP + TN) / (total + SMALL)
    ax.set_title(
        f"Confusion Matrix  |  ResNet-50 + SAM\n"
        f"Accuracy: {acc:.4f}  ({TP + TN} / {total})",
        fontsize=14, fontweight="bold", color="#1C2B3A", pad=18)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
                dpi=200, bbox_inches="tight", facecolor="#F8FAFC")
    plt.close()


def plot_accuracy_vs_interpretability(records: List[Dict], opt_thr: float) -> None:
    CHUNK  = 30
    chunks = [records[i:i + CHUNK] for i in range(0, len(records), CHUNK)
              if len(records[i:i + CHUNK]) >= 5]
    if not chunks:
        return

    acc_vals, mrs_vals = [], []
    for ch in chunks:
        preds  = np.array([1 if r["prob"] >= opt_thr else 0 for r in ch])
        labels = np.array([r["true_class"] for r in ch])
        acc_vals.append(float((preds == labels).mean()))
        mrs_vals.append(float(np.mean([r["Ri"] for r in ch])))

    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(acc_vals, mrs_vals, s=55, c=acc_vals, cmap="coolwarm",
                    edgecolors="white", linewidths=0.5, alpha=0.88, vmin=0.0, vmax=1.0)
    plt.colorbar(sc, ax=ax, label="Accuracy")
    ax.set_xlabel("Chunk Accuracy", fontsize=12)
    ax.set_ylabel("Interpretability (MRS)", fontsize=12)
    ax.set_title("Accuracy vs Interpretability  |  ResNet-50 + SAM",
                 fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "accuracy_vs_interpretability.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
#  SAM ATTENTION MAP EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_sam_attention(x_img: np.ndarray,
                          attn_extractor: keras.Model) -> Tuple[np.ndarray, np.ndarray]:
    """
    Given a preprocessed image array (H×W×3, values in [0,1]),
    return:
        alpha_vis  — visualisation-ready heatmap, shape (H, W), range [0,1]
        raw_attn   — raw attention map (7×7), upsampled to (H, W), range [0,1]
    """
    x = np.asarray(x_img, dtype=np.float32)
    if x.ndim == 3:
        x = x[np.newaxis]
    if float(x.max()) > 1.0 + 1e-3:
        x = x / 255.0

    attn_7x7, _ = attn_extractor(x, training=False)     # (1, 7, 7, 1)
    attn_7x7    = attn_7x7[0, :, :, 0].numpy()          # (7, 7)

    H, W = x_img.shape[:2]
    raw_attn  = cv2.resize(attn_7x7, (W, H), interpolation=cv2.INTER_LINEAR)
    raw_attn  = np.clip(raw_attn, 0.0, None)
    attn_max  = raw_attn.max()
    if attn_max > SMALL:
        raw_attn = raw_attn / attn_max

    # Mild Gaussian smoothing for display clarity
    ks        = max(3, int(np.ceil(6.0 * 4.0))) | 1     # σ=4 for 224-px image
    alpha_vis = cv2.GaussianBlur(raw_attn, (ks, ks), sigmaX=4.0, sigmaY=4.0)
    amin, amax = alpha_vis.min(), alpha_vis.max()
    if amax - amin > SMALL:
        alpha_vis = (alpha_vis - amin) / (amax - amin)

    return alpha_vis.astype(np.float32), raw_attn.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
#  VISUALISATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def overlay_attention(img_rgb: np.ndarray, alpha: np.ndarray,
                      strength: float = 0.65) -> np.ndarray:
    """Blend attention heatmap (jet colormap) over the original X-ray."""
    cmap  = plt.get_cmap("jet")
    rgba  = cmap(alpha).astype(np.float32)
    rgba[..., 3] = 0.28 * strength + (strength - 0.28 * strength) * alpha
    heat  = rgba[..., :3]
    a_map = rgba[..., 3:4]
    return np.clip(a_map * heat + (1.0 - a_map) * img_rgb, 0.0, 1.0).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
#  ADDITIONAL SAM-SPECIFIC OVERVIEW PLOT
# ─────────────────────────────────────────────────────────────────────────────

def plot_sam_attention_overview(samples_normal: List[Dict],
                                samples_pneumonia: List[Dict],
                                n_each: int = 3) -> None:
    """
    Grid: rows = selected samples, cols = [Original | SAM Attention | Overlay]
    Half normal, half pneumonia, with class label strip.
    """
    selected = (samples_normal[:n_each] + samples_pneumonia[:n_each])
    n_rows   = len(selected)
    if n_rows == 0:
        return

    BG, TITLE_COL, SUB_COL = "#F7F9FC", "#1C2B3A", "#4A5568"

    fig = plt.figure(figsize=(13, 4.2 * n_rows + 1.0))
    fig.patch.set_facecolor(BG)
    gs  = GridSpec(n_rows + 1, 3, figure=fig,
                   height_ratios=[0.10] + [1.0] * n_rows,
                   hspace=0.08, wspace=0.06)

    col_labels = ["Original X-Ray", "SAM Attention Map", "Attention Overlay"]
    for ci, txt in enumerate(col_labels):
        ax = fig.add_subplot(gs[0, ci])
        ax.set_facecolor(BG)
        ax.axhline(y=0.18, xmin=0.1, xmax=0.9, color="#00ACC1",
                   linewidth=3, solid_capstyle="round")
        ax.text(0.5, 0.65, txt, ha="center", va="center", color=TITLE_COL,
                fontsize=12, fontweight="bold", transform=ax.transAxes)
        ax.axis("off")

    for row, s in enumerate(selected):
        img        = s["img"]
        alpha_disp = s["alpha"]
        prob       = s["prob"]
        gt         = "NORMAL" if s["true_class"] == 0 else "PNEUMONIA"
        pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
        correct    = gt == pred
        sym        = "✓" if correct else "✗"
        sym_col    = "#2E7D32" if correct else "#C62828"
        hdr_col    = "limegreen" if s["true_class"] == 0 else "#FF69B4"
        title_str  = f"GT: {gt}   Pred: {pred}   p(PNEU): {prob:.3f}"

        ax0 = fig.add_subplot(gs[row + 1, 0])
        ax0.imshow(img, interpolation="bilinear"); ax0.axis("off")
        ax0.set_title(title_str, color=TITLE_COL, fontsize=7.5, pad=4)
        ax0.text(0.97, 0.97, sym, transform=ax0.transAxes, ha="right", va="top",
                 fontsize=14, fontweight="bold", color=sym_col,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=sym_col,
                           lw=1.5, alpha=0.85))
        ax0.spines["bottom"].set_color(hdr_col); ax0.spines["bottom"].set_linewidth(3)

        ax1 = fig.add_subplot(gs[row + 1, 1])
        im = ax1.imshow(alpha_disp, cmap="jet", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
        ax1.axis("off")
        ax1.set_title("SAM Attention", color=SUB_COL, fontsize=7.5, pad=4)
        cbar = fig.colorbar(im, ax=ax1, fraction=0.040, pad=0.02)
        cbar.set_ticks([0.0, 0.5, 1.0])
        cbar.set_ticklabels(["Low", "Mid", "High"])
        cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=6)
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
        cbar.outline.set_edgecolor("#CCCCCC")

        ax2 = fig.add_subplot(gs[row + 1, 2])
        ax2.imshow(overlay_attention(img, alpha_disp, strength=0.65),
                   interpolation="bilinear")
        ax2.axis("off")
        ax2.set_title("Overlay", color=SUB_COL, fontsize=7.5, pad=4)

    fig.suptitle("SAM Attention Maps  |  ResNet-50 + SAM  (Normal & Pneumonia)",
                 fontsize=12, fontweight="bold", color=TITLE_COL, y=1.002)
    path = os.path.join(OUTPUT_DIR, "sam_attention_maps.png")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"SAM overview saved  →  {path}")


# ─────────────────────────────────────────────────────────────────────────────
#  EXPLANATION PANELS  (mirrored from Grad-CAM pipeline)
# ─────────────────────────────────────────────────────────────────────────────

def _render_class_panel(samples: List[Dict], class_label: str,
                         header_color: str, save_path: str) -> None:
    n = len(samples)
    if n == 0:
        return

    BG, CARD_BG   = "#F7F9FC", "#FFFFFF"
    TITLE_COL, SUB_COL = "#1C2B3A", "#4A5568"

    fig = plt.figure(figsize=(13, 4.4 * n + 1.1))
    fig.patch.set_facecolor(BG)
    gs  = GridSpec(n + 1, 3, figure=fig,
                   height_ratios=[0.12] + [1.0] * n,
                   hspace=0.08, wspace=0.06)

    col_labels = ["Original X-Ray", "SAM Attention Map", "Attention Overlay"]
    for ci, txt in enumerate(col_labels):
        ax = fig.add_subplot(gs[0, ci])
        ax.set_facecolor(BG)
        ax.axhline(y=0.18, xmin=0.1, xmax=0.9, color=header_color,
                   linewidth=3, solid_capstyle="round")
        ax.text(0.5, 0.65, txt, ha="center", va="center", color=TITLE_COL,
                fontsize=12, fontweight="bold", transform=ax.transAxes)
        ax.text(0.5, 0.30, class_label, ha="center", va="center",
                color=header_color, fontsize=10, fontweight="bold",
                transform=ax.transAxes)
        ax.axis("off")

    for row, s in enumerate(samples):
        img        = s["img"]
        alpha_disp = s["alpha"]
        prob       = s["prob"]
        pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
        correct    = class_label == pred
        sym        = "✓" if correct else "✗"
        sym_col    = "#2E7D32" if correct else "#C62828"
        title_str  = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

        ax0 = fig.add_subplot(gs[row + 1, 0])
        ax0.set_facecolor(CARD_BG)
        ax0.imshow(img, interpolation="bilinear"); ax0.axis("off")
        ax0.set_title(title_str, color=TITLE_COL, fontsize=7.5, pad=4, loc="center")
        ax0.text(0.97, 0.97, sym, transform=ax0.transAxes, ha="right", va="top",
                 fontsize=14, fontweight="bold", color=sym_col,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=sym_col, lw=1.5, alpha=0.85))

        ax1 = fig.add_subplot(gs[row + 1, 1])
        ax1.set_facecolor(CARD_BG)
        im = ax1.imshow(alpha_disp, cmap="jet", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
        ax1.axis("off")
        ax1.set_title("SAM Attention", color=SUB_COL, fontsize=7.5, pad=4)
        cbar = fig.colorbar(im, ax=ax1, fraction=0.040, pad=0.02)
        cbar.set_ticks([0.0, 0.5, 1.0])
        cbar.set_ticklabels(["Low", "Mid", "High"])
        cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=6)
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
        cbar.outline.set_edgecolor("#CCCCCC")

        ax2 = fig.add_subplot(gs[row + 1, 2])
        ax2.set_facecolor(CARD_BG)
        ax2.imshow(overlay_attention(img, alpha_disp, strength=0.65),
                   interpolation="bilinear")
        ax2.axis("off")
        ax2.set_title("Overlay", color=SUB_COL, fontsize=7.5, pad=4)

    fig.suptitle(f"{class_label}  |  ResNet-50 + SAM",
                 fontsize=12, fontweight="bold", color=TITLE_COL, y=1.002)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Panel saved  →  {save_path}")


def _render_single_sample(s: Dict, class_label: str,
                            header_color: str, save_path: str) -> None:
    BG, CARD_BG   = "#F7F9FC", "#FFFFFF"
    TITLE_COL, SUB_COL = "#1C2B3A", "#4A5568"

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    fig.patch.set_facecolor(BG)
    for ax in axes:
        ax.set_facecolor(CARD_BG)

    img        = s["img"]
    alpha_disp = s["alpha"]
    prob       = s["prob"]
    pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
    correct    = class_label == pred
    sym        = "✓" if correct else "✗"
    sym_col    = "#2E7D32" if correct else "#C62828"
    title_str  = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

    axes[0].imshow(img, interpolation="bilinear"); axes[0].axis("off")
    axes[0].set_title(f"Original X-Ray\n{title_str}", color=TITLE_COL, fontsize=9, pad=4)
    axes[0].text(0.97, 0.97, sym, transform=axes[0].transAxes,
                 ha="right", va="top", fontsize=15, fontweight="bold", color=sym_col,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=sym_col, lw=1.5, alpha=0.85))

    im = axes[1].imshow(alpha_disp, cmap="jet", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
    axes[1].axis("off")
    axes[1].set_title("SAM Attention Map", color=SUB_COL, fontsize=9, pad=4)
    cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_ticks([0.0, 0.5, 1.0])
    cbar.set_ticklabels(["Low", "Mid", "High"])
    cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=7)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
    cbar.outline.set_edgecolor("#CCCCCC")

    axes[2].imshow(overlay_attention(img, alpha_disp, strength=0.65),
                   interpolation="bilinear")
    axes[2].axis("off")
    axes[2].set_title("Attention Overlay", color=SUB_COL, fontsize=9, pad=4)

    fig.suptitle(f"{class_label}  |  ResNet-50 + SAM",
                 color=TITLE_COL, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Single sample  →  {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
#  SAM ANALYSIS  (collect panels + per-sample attention maps)
# ─────────────────────────────────────────────────────────────────────────────

def run_sam_analysis(test_gen, attn_extractor: keras.Model,
                     all_probs: np.ndarray, opt_thr: float,
                     n_per_class: int = 6) -> Tuple[List[Dict], List[Dict]]:
    """
    Iterate the test set, extract SAM attention maps for n_per_class samples
    per class, produce explanation panels and individual sample figures.
    Returns (panel_normal, panel_pneumonia).
    """
    test_gen.reset()
    panel_normal: List[Dict]    = []
    panel_pneumonia: List[Dict] = []
    sample_idx = 0

    for batch_x, batch_y in test_gen:
        for i in range(len(batch_x)):
            if sample_idx >= len(all_probs):
                break
            true_class = int(batch_y[i])
            prob       = float(all_probs[sample_idx])
            sample_idx += 1

            need = ((true_class == 0 and len(panel_normal)    < n_per_class) or
                    (true_class == 1 and len(panel_pneumonia) < n_per_class))
            if not need:
                continue

            x_img      = batch_x[i]
            alpha_vis, _ = extract_sam_attention(x_img, attn_extractor)

            entry = dict(img=x_img, alpha=alpha_vis, prob=prob, true_class=true_class)
            if true_class == 0:
                panel_normal.append(entry)
            else:
                panel_pneumonia.append(entry)

        if (len(panel_normal) >= n_per_class and
                len(panel_pneumonia) >= n_per_class):
            break

    # ── Class panels ──────────────────────────────────────────────────────────
    _render_class_panel(panel_normal, "NORMAL", "limegreen",
                        os.path.join(OUTPUT_DIR, "explanation_normal_panel.png"))
    _render_class_panel(panel_pneumonia, "PNEUMONIA", "#FF69B4",
                        os.path.join(OUTPUT_DIR, "explanation_pneumonia_panel.png"))

    # ── Individual sample figures ─────────────────────────────────────────────
    for idx, s in enumerate(panel_normal[:5]):
        _render_single_sample(s, "NORMAL", "limegreen",
                              os.path.join(OUTPUT_DIR, f"single_normal_{idx + 1}.png"))
    for idx, s in enumerate(panel_pneumonia[:5]):
        _render_single_sample(s, "PNEUMONIA", "#FF69B4",
                              os.path.join(OUTPUT_DIR, f"single_pneumonia_{idx + 1}.png"))

    # ── Combined SAM overview (mixed classes) ─────────────────────────────────
    plot_sam_attention_overview(panel_normal, panel_pneumonia, n_each=3)

    return panel_normal, panel_pneumonia


# ─────────────────────────────────────────────────────────────────────────────
#  MRS COMPUTATION  (using SAM attention maps as relevance signal)
# ─────────────────────────────────────────────────────────────────────────────

def compute_mrs(test_gen, attn_extractor: keras.Model,
                all_probs: np.ndarray, opt_thr: float) -> Dict:
    """
    MRS formula (Colin & Surantha, 2025 / Hedström et al. 2023):
        Ri  = Σ(A_pos · M) / (Σ A_pos + ε)
        MRS = (1/N) Σ Ri
    where A_pos = max(attention_map, 0) and M = binary lung mask.
    """
    print(f"\n{'='*60}\nCOMPUTING MRS  (SAM attention)\n{'='*60}")
    test_gen.reset()
    records: List[Dict] = []
    sample_idx = 0
    n_failed   = 0

    for batch_x, batch_y in test_gen:
        for i in range(len(batch_x)):
            if sample_idx >= len(all_probs):
                break
            true_class = int(batch_y[i])
            prob       = float(all_probs[sample_idx])
            x_img      = batch_x[i]
            sample_idx += 1

            try:
                _, raw_attn = extract_sam_attention(x_img, attn_extractor)
            except Exception as exc:
                print(f"  WARNING sample {sample_idx}: {exc}")
                n_failed += 1
                continue

            Ri, lung_cov = compute_ri(x_img, raw_attn)
            records.append(dict(
                sample_id        = sample_idx - 1,
                true_class       = true_class,
                prob             = prob,
                pred_class       = 1 if prob >= opt_thr else 0,
                Ri               = Ri,
                lung_pixel_ratio = lung_cov,
                attn_max         = float(raw_attn.max())))

            if sample_idx % 50 == 0:
                running = np.mean([r["Ri"] for r in records])
                ok_flag = "✓" if LUNG_ADAPTIVE_LO <= lung_cov <= LUNG_ADAPTIVE_HI else "⚠"
                print(f"  [{sample_idx:>4}/{test_gen.n}]  MRS={running:.4f}  "
                      f"Ri={Ri:.4f}  cov={lung_cov:.2f}{ok_flag}")

        if sample_idx >= len(all_probs):
            break

    if not records:
        print("ERROR: No MRS records produced.")
        return {}

    all_ri  = np.array([r["Ri"] for r in records])
    ri_pneu = np.array([r["Ri"] for r in records if r["true_class"] == 1])
    ri_norm = np.array([r["Ri"] for r in records if r["true_class"] == 0])
    all_cov = np.array([r["lung_pixel_ratio"] for r in records])

    MRS_overall   = float(all_ri.mean())
    MRS_pneumonia = float(ri_pneu.mean()) if len(ri_pneu) else 0.0
    MRS_normal    = float(ri_norm.mean()) if len(ri_norm) else 0.0
    MRS_std       = float(all_ri.std())
    delta         = MRS_overall - MRS_BASELINE

    # Trade-off score (Colin & Surantha, 2025 — Equation 11)
    # Trade-off = Accuracy − λ(1 − MRS), using λ = 1 (equal weight)
    # Accuracy placeholder; caller will substitute actual value
    trade_off_placeholder = None

    print(f"\n{'='*60}\nMEAN RELEVANCE SCORE\n{'='*60}")
    print(f"  MRS Overall   : {MRS_overall:.4f}  ±{MRS_std:.4f}")
    print(f"  MRS PNEUMONIA : {MRS_pneumonia:.4f}  (n={len(ri_pneu)})")
    print(f"  MRS NORMAL    : {MRS_normal:.4f}  (n={len(ri_norm)})")
    print(f"  Avg coverage  : {all_cov.mean():.3f}")
    print(f"  Δ vs baseline : {'+' if delta >= 0 else ''}{delta:.4f}")
    if n_failed:
        print(f"  Failed samples: {n_failed}")

    # Save per-sample CSV
    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

    # ── MRS histogram (dual panel — mirrors Grad-CAM pipeline) ───────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(all_ri, bins=30, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].axvline(MRS_overall,  color="tomato", lw=2, label=f"MRS={MRS_overall:.4f}")
    axes[0].axvline(MRS_BASELINE, color="orange", lw=1.5, linestyle="--",
                    label=f"Baseline={MRS_BASELINE:.2f}")
    axes[0].set_xlabel("Ri"); axes[0].set_ylabel("Count")
    axes[0].set_title("MRS — All Test Samples  (SAM)")
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].hist(ri_pneu, bins=20, color="#FF69B4", edgecolor="white", alpha=0.75,
                 label=f"PNEUMONIA  {MRS_pneumonia:.4f}")
    axes[1].hist(ri_norm, bins=20, color="limegreen", edgecolor="white", alpha=0.75,
                 label=f"NORMAL  {MRS_normal:.4f}")
    axes[1].axvline(MRS_BASELINE, color="orange", lw=1.5, linestyle="--",
                    label=f"Baseline={MRS_BASELINE:.2f}")
    axes[1].set_xlabel("Ri"); axes[1].set_ylabel("Count")
    axes[1].set_title("MRS — Per Class  (SAM)")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.suptitle(
        f"MRS Histogram  |  ResNet-50 + SAM  |  MRS={MRS_overall:.4f}",
        fontweight="bold", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "mrs_histogram.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return dict(
        MRS_overall         = round(MRS_overall,   4),
        MRS_pneumonia       = round(MRS_pneumonia, 4),
        MRS_normal          = round(MRS_normal,    4),
        MRS_std             = round(MRS_std,        4),
        n_samples_mrs       = len(records),
        n_failed_sam        = n_failed,
        baseline_paper      = MRS_BASELINE,
        delta_vs_baseline   = round(delta, 4),
        avg_lung_coverage   = round(float(all_cov.mean()), 4))


# ─────────────────────────────────────────────────────────────────────────────
#  ZIP ALL OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────

def zip_outputs(output_dir: str, zip_name: str = "sam_results.zip") -> str:
    zip_path = os.path.join(output_dir, zip_name)
    image_exts = {".png", ".json", ".csv"}
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(output_dir)):
            if fname == zip_name:
                continue
            ext = os.path.splitext(fname)[1].lower()
            if ext in image_exts:
                fpath = os.path.join(output_dir, fname)
                zf.write(fpath, arcname=fname)
    print(f"\nAll outputs zipped  →  {zip_path}")
    return zip_path


# ─────────────────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print(f"{'='*70}")
    print(f"  ResNet-50 + SAM  |  Chest X-ray Pneumonia Classification")
    print(f"  EPOCHS={EPOCHS}  LR={LR}  L2={L2_VALUE}  SAM_KERNEL={SAM_KERNEL_SIZE}")
    print(f"  Reference: Colin & Surantha, Information 2025, 16, 53")
    print(f"{'='*70}\n")

    # ── Data ─────────────────────────────────────────────────────────────────
    train_gen, val_gen, test_gen, _, steps = get_loaders(DATA_ROOT)
    class_weight_dict = compute_class_weights_from_gen(train_gen)

    # ── Model ─────────────────────────────────────────────────────────────────
    ckpt = os.path.join(OUTPUT_DIR, "best_model.keras")
    custom_objs = {"SpatialAttentionModule": SpatialAttentionModule}

    if os.path.exists(ckpt):
        print(f"Loading checkpoint  ←  {ckpt}")
        model = keras.models.load_model(ckpt, custom_objects=custom_objs)
        history = None
        hist_path = os.path.join(OUTPUT_DIR, "history.json")
        if os.path.exists(hist_path):
            history = json.load(open(hist_path))
    else:
        model, _ = build_resnet50_sam()
        model.summary(line_length=100)
        model, history = train_model(model, train_gen, val_gen, steps, class_weight_dict)
        with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as fh:
            json.dump(history, fh, indent=2)

    # Re-build attention extractor from the (possibly loaded) model
    attn_extractor = keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer("sam").output[1],   # (B,7,7,1) attention map
                   model.output],                        # logit
        name    = "AttentionExtractor")

    # ── Training plots ─────────────────────────────────────────────────────
    if history:
        plot_training_accuracy(history)
        plot_training_loss(history)

    # ── Evaluation ─────────────────────────────────────────────────────────
    metrics, opt_thr = evaluate_model(model, val_gen, test_gen)

    # ── Collect test probabilities once ─────────────────────────────────────
    test_gen.reset()
    all_probs = tf.sigmoid(
        model.predict(test_gen, verbose=1).ravel()
    ).numpy()[:test_gen.n]

    # ── SAM visual analysis ─────────────────────────────────────────────────
    test_gen.reset()
    panel_normal, panel_pneumonia = run_sam_analysis(
        test_gen, attn_extractor, all_probs, opt_thr, n_per_class=6)

    # ── MRS ─────────────────────────────────────────────────────────────────
    test_gen.reset()
    mrs_results = compute_mrs(test_gen, attn_extractor, all_probs, opt_thr)

    # ── Metric plots ─────────────────────────────────────────────────────────
    mrs_overall = mrs_results.get("MRS_overall", 0.0)
    plot_metrics_bar_chart(metrics, mrs_overall)
    plot_ppv_npv(metrics)
    plot_confusion_matrix(metrics)

    # ── Accuracy vs Interpretability scatter ─────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    if os.path.exists(csv_path):
        records_list = pd.read_csv(csv_path).to_dict("records")
        plot_accuracy_vs_interpretability(records_list, opt_thr)

    # ── Trade-off score (Colin & Surantha, 2025 — Eq. 11, λ=1) ───────────────
    trade_off = round(metrics["accuracy"] - 1.0 * (1.0 - mrs_overall), 4)
    print(f"\n  Trade-off Score (λ=1): {trade_off:.4f}")

    # ── Final report ─────────────────────────────────────────────────────────
    final = {
        **{k: v for k, v in metrics.items() if k not in ("TP", "TN", "FP", "FN")},
        **mrs_results,
        "trade_off_score"              : trade_off,
        "interpretability_technique"   : "Spatial Attention Mechanism (SAM)",
        "sam_formula"                  : "A = softmax(Conv(F));  F' = A ⊙ F",
        "sam_kernel_size"              : SAM_KERNEL_SIZE,
        "sam_reference"                : "Colin & Surantha, Information 2025, 16, 53",
        "mrs_formula"                  : "Ri = Σ(A_pos·M)/(ΣA_pos+ε)  [Hedström et al. 2023]",
        "threshold_source"             : "validation set only",
        "epochs_config"                : EPOCHS,
        "lr"                           : LR,
        "l2"                           : L2_VALUE,
    }
    with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in final.items()}, f, indent=2)

    # ── Post-run diagnostic ───────────────────────────────────────────────────
    avg_cov = mrs_results.get("avg_lung_coverage", 0.0)
    delta   = mrs_results.get("delta_vs_baseline", 0.0)
    print(f"\n{'─'*50}  POST-RUN DIAGNOSTICS")
    if avg_cov < LUNG_ADAPTIVE_LO:
        print(f"⚠  cov={avg_cov:.3f} < floor — consider reducing LUNG_BOTTOM_PAD")
    elif avg_cov > LUNG_ADAPTIVE_HI:
        print(f"⚠  cov={avg_cov:.3f} > ceil  — consider increasing LUNG_BOTTOM_PAD")
    else:
        print(f"✓  cov={avg_cov:.3f}  in [{LUNG_ADAPTIVE_LO}, {LUNG_ADAPTIVE_HI}]")

    if   mrs_overall >= MRS_BASELINE:
        print(f"✓  MRS={mrs_overall:.4f} ≥ paper baseline {MRS_BASELINE}")
    elif mrs_overall >= 0.10:
        print(f"△  MRS={mrs_overall:.4f}  — review SAM kernel or coverage")
    else:
        print(f"✗  MRS={mrs_overall:.4f}  — check attention layer output")
    print(f"   Δ vs baseline ({MRS_BASELINE}): {'+' if delta >= 0 else ''}{delta:.4f}")
    print(f"   Trade-off Score (λ=1) : {trade_off:.4f}")

    zip_outputs(OUTPUT_DIR)
    print(f"\nAll outputs  →  {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

2026-05-19 05:07:21.646917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779167241.839795      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779167241.899142      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779167242.320767      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779167242.320807      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779167242.320809      57 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: 2
  ResNet-50 + SAM  |  Chest X-ray Pneumonia Classification
  EPOCHS=50  LR=0.001  L2=0.0001  SAM_KERNEL=7
  Reference: Colin & Surantha, Information 2025, 16, 53

Found 4708 validated image filenames belonging to 2 classes.
Found 524 validated image filenames belonging to 2 classes.
Found 624 validated image filenames belonging to 2 classes.
Train=4708  Val=524  Test=624
Class map: {'NORMAL': 0, 'PNEUMONIA': 1}


I0000 00:00:1779167273.220609      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779167273.226661      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "ResNet50_SAM_Pneumonia"

...

 Trainable params: 23,636,996 (90.17 MB)

 Non-trainable params: 53,122 (207.51 KB)

Epoch 1/50


I0000 00:00:1779167317.297904     131 service.cc:152] XLA service 0x7b5708003220 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779167317.297960     131 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779167317.297967     131 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779167323.964546     131 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779167351.296410     131 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6895 - auc: 0.8633 - loss: 0.9416
Epoch 1: val_loss improved from inf to 6.45225, saving model to ./resnet50_sam_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 291s 1s/step - accuracy: 0.6903 - auc: 0.8637 - loss: 0.9408 - val_accuracy: 0.7424 - val_auc: 0.5000 - val_loss: 6.4523 - learning_rate: 0.0010
Epoch 2/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 925ms/step - accuracy: 0.8839 - auc: 0.9313 - loss: 0.6494
Epoch 2: val_loss improved from 6.45225 to 0.94268, saving model to ./resnet50_sam_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - accuracy: 0.8840 - auc: 0.9313 - loss: 0.6490 - val_accuracy: 0.7424 - val_auc: 0.4134 - val_loss: 0.9427 - learning_rate: 0.0010
Epoch 3/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 917ms/step - accuracy: 0.8726 - auc: 0.9252 - loss: 0.5690
Epoch 3: val_loss improved from 0.94268 to 0.87575, saving model to ./resnet50_sam_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 147s 995ms/ste